# Using Project Source Code

This chapter demonstrates how to import library code from `src/` without
`sys.path` hacks. The project is installed in editable mode when you run
`uv sync`, so notebooks can import packages directly.

The example pipeline:

1. Generate synthetic data with a small simulation
2. Bundle a Random Forest and an SVM (with `StandardScaler`) into a single `MultiRegressor` composite estimator
3. Hand that composite to `compare_models`, which fits each base, computes bootstrap confidence intervals, MAPIE split conformal prediction intervals, and bootstrap-CI'd regression metrics
4. Visualize the comparison with faceted Altair charts

## Imports

No path bootstrapping is required — the packages are installed by `uv sync`.

In [ ]:
from sklearn.model_selection import train_test_split

from analysis import compare_models
from core import ModelKind, Settings, TrainingData, build_split_dataset
from prediction import (
    MultiRegressor,
    random_forest_regressor,
    regression_pipeline,
    svm_regressor,
)
from simulation import generate_dataset
from visualization import (
    plot_dataset,
    plot_interval_metrics,
    plot_intervals,
    plot_regression_metrics,
)

## Generate synthetic data

In [ ]:
settings = Settings(n_samples=5000, seed=0, svm_gamma=0.025)
data = generate_dataset(settings)
data.head()

In [ ]:
plot_dataset(data)

## Split into train, calibration, and test sets

In [ ]:
train, remainder = train_test_split(data, test_size=0.3, random_state=0)
calib, test = train_test_split(remainder, test_size=0.5, random_state=0)
split_data = build_split_dataset(
    TrainingData.validate(train),
    TrainingData.validate(calib),
    TrainingData.validate(test),
)
len(train), len(calib), len(test)

## Build the composite estimator

Each model is wired through `regression_pipeline(...)` to inherit the polynomial + Fourier feature
expansion. The SVM factory additionally wraps `SVR` behind a `StandardScaler` (an inner pipeline),
so scaling happens after feature expansion and only for the model that needs it.

`MultiRegressor` is a sklearn-native composite (`BaseEstimator + RegressorMixin + TransformerMixin`).
`.transform(X)` returns per-base predictions as a `(n_samples, n_estimators)` matrix, and the unfitted
estimator specs stay introspectable via `.estimators` — which is what `bootstrap_confidence_intervals`
and `fit_conformal` need to `clone()` per base.

In [ ]:
random_forest_pipeline = regression_pipeline(random_forest_regressor(settings), settings)
random_forest_pipeline

In [ ]:
svm_pipeline = regression_pipeline(svm_regressor(settings), settings)
svm_pipeline

In [ ]:
regressors = MultiRegressor(
    estimators=[
        (ModelKind.RANDOM_FOREST.value, random_forest_pipeline),
        (ModelKind.SVM.value, svm_pipeline),
    ],
)
regressors

## Fit, calibrate, score, and bootstrap in one call

`compare_models` walks each `(name, pipeline)` pair in the composite once and returns a
`ModelComparisonReport` dataclass with six tagged DataFrames:

- `predictions` — per-model point predictions with ground truth
- `confidence` — bootstrap confidence intervals (refit-on-resample) per model
- `prediction` — MAPIE split conformal prediction intervals per model
- `regression_metrics` — RMSE/MAE/R² with bootstrap CIs, per model
- `confidence_metrics` and `prediction_metrics` — interval width and MWI scores, per model

All concatenation and `model`-column tagging happens inside `src/`; the notebook stays declarative.

In [ ]:
report = compare_models(split_data, regressors, settings)
report.predictions.head()

## Visualize the comparison

`plot_intervals` renders one small-multiples panel per model: the data scatter,
the bootstrap confidence band, the conformal prediction band, and the regression line.

In [ ]:
plot_intervals(data, report)

## Pipeline evaluation

`plot_regression_metrics` facets by metric so each metric (RMSE, MAE, R²) gets its own y-axis —
comparing models across metrics on a shared scale would be misleading because the metrics live in
different units. Models sit on the x-axis within each facet, and each error bar shows the
bootstrap CI.

`plot_interval_metrics` facets by interval kind (confidence vs. prediction). Within each facet,
the metrics (width, MWI) are grouped on the x-axis and the models are color-coded — so the chart
compares confidence-against-confidence and prediction-against-prediction, not CI-against-PI.

In [ ]:
plot_regression_metrics(report)

In [ ]:
plot_interval_metrics(report)

In [ ]:
report.regression_metrics

In [ ]:
report.confidence_metrics

In [ ]:
report.prediction_metrics

## Tests

Each module under `src/` has a matching test module under `tests/`.
Run the full suite with:

```bash
uv run poe test
```

CI runs tests before building the book (`uv run poe ci`).